# 04 — Streaming

Soma supports processing data in **chunks** (streaming mode). This is essential
for inference pipelines where data arrives continuously.

Each filter declares a **StreamMode** that tells the runtime how its state
behaves during streaming:

| Mode | State behavior | Example |
|---|---|---|
| `fixed` (default) | State from `fit()` is frozen — never changes | Standard scaler, trained model |
| `evolving` | State updates with each chunk (checkpointed) | Running average, online learning |
| `barrier` | Accumulates all chunks, then processes at once | Sorting, global aggregation |

> **Note:** The stream executor runs in the Rust runtime. From Python, you
> control streaming behavior by setting `_stream_mode` on your filter classes.

## 4.1 — FixedState: the most common mode

A `fixed` filter uses the state learned during `fit()` without modification.
Each chunk is processed independently with the same state.

```
fit(training_data) → state (frozen)
chunk_1 → forward(chunk_1, state) → result_1
chunk_2 → forward(chunk_2, state) → result_2
chunk_3 → forward(chunk_3, state) → result_3
```

This is the default and the most cacheable: same chunk + same state = same result.

In [ ]:
from soma import Filter, Pipeline

class StandardScaler(Filter):
    """Normalizes using statistics learned during training.
    State is fixed — the same mean/std is used for every chunk."""
    _stream_mode = "fixed"  # default, but explicit for clarity

    def fit(self, x, y=None):
        mean = sum(x) / len(x)
        std = (sum((v - mean) ** 2 for v in x) / len(x)) ** 0.5
        return {"mean": mean, "std": std}

    def forward(self, x, state):
        return [(v - state["mean"]) / max(state["std"], 1e-8) for v in x]

# Train on batch data
scaler = StandardScaler()
state = scaler.fit([100.0, 200.0, 300.0, 400.0, 500.0])
print(f"Learned state: mean={state['mean']}, std={state['std']:.2f}")

# Simulate streaming: process each chunk with the SAME frozen state
chunks = [[150.0, 250.0], [350.0, 450.0], [550.0]]
for i, chunk in enumerate(chunks):
    result = scaler.forward(chunk, state)
    print(f"Chunk {i}: {chunk} → {[f'{v:.2f}' for v in result]}")

## 4.2 — Evolving: state that grows with each chunk

An `evolving` filter's state **updates** after processing each chunk.
The runtime checkpoints the state periodically for fault tolerance.

```
fit(training_data) → state_0
chunk_1 → forward(chunk_1, state_0) → result_1, state_1
chunk_2 → forward(chunk_2, state_1) → result_2, state_2  (checkpoint)
chunk_3 → forward(chunk_3, state_2) → result_3, state_3
```

Use case: running statistics, online learning, accumulating embeddings.

In [ ]:
class RunningMean(Filter):
    """Computes a running mean across chunks.
    State evolves: count and sum accumulate with each chunk."""
    _stream_mode = "evolving"

    def fit(self, x, y=None):
        # Initial state
        return {"count": 0, "sum": 0.0, "mean": 0.0}

    def forward(self, x, state):
        # Update running statistics
        new_count = state["count"] + len(x)
        new_sum = state["sum"] + sum(x)
        new_mean = new_sum / new_count

        # In the Rust runtime, the returned dict would update the state.
        # Here we simulate by mutating state directly.
        state["count"] = new_count
        state["sum"] = new_sum
        state["mean"] = new_mean

        # Output: each value minus the running mean
        return [v - new_mean for v in x]

rm = RunningMean()
state = rm.fit([])

# Simulate streaming chunks
chunks = [[10.0, 20.0], [30.0, 40.0], [100.0]]
for i, chunk in enumerate(chunks):
    result = rm.forward(chunk, state)
    print(f"Chunk {i}: {chunk}")
    print(f"  Running mean: {state['mean']:.1f} (n={state['count']})")
    print(f"  Output:       {[f'{v:.1f}' for v in result]}")

## 4.3 — Barrier: collect everything, then process

A `barrier` filter **materializes** all chunks before processing.
It waits for the entire stream to arrive, then runs `forward()` on the full dataset.

```
chunk_1 → buffer
chunk_2 → buffer
chunk_3 → buffer
         → forward([all_chunks], state) → result
```

Use case: sorting, global statistics, operations that need the full dataset.

In [ ]:
class Sorter(Filter):
    """Sorts data globally — needs all chunks before it can produce output."""
    _stream_mode = "barrier"

    def fit(self, x, y=None):
        return {}

    def forward(self, x, state):
        return sorted(x)

# Simulate barrier behavior: accumulate chunks, then process
sorter = Sorter()
state = sorter.fit([])

chunks = [[30.0, 10.0], [50.0, 20.0], [40.0]]
print("Incoming chunks:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk}")

# Barrier: collect all chunks first
all_data = [v for chunk in chunks for v in chunk]
print(f"\nMaterialized: {all_data}")

# Then process the full dataset
result = sorter.forward(all_data, state)
print(f"Sorted:       {result}")

## 4.4 — Combining stream modes in a pipeline

A real streaming pipeline might mix modes:

```
[FixedState scaler] → [Evolving running_mean] → [Barrier sorter]
     per-chunk            state updates              waits for all
```

The Rust runtime handles the orchestration: it knows which filters can
process chunks immediately, which need state updates, and which must wait.

In [ ]:
# Build a mixed-mode pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),      # fixed: uses frozen mean/std
    ("running_mean", RunningMean()),    # evolving: updates running stats
])

# Train the pipeline
training_data = [10.0, 20.0, 30.0, 40.0, 50.0]
pipeline.fit(training_data)
print("Pipeline fitted with training data")
print(f"Filters: {pipeline.filter_names()}")

# In batch mode (predict), the full data goes through at once
result = pipeline.predict([15.0, 25.0, 35.0, 45.0])
print(f"\nBatch predict: {[f'{v:.2f}' for v in result]}")

# In streaming mode (handled by Rust runtime), chunks would flow through
# respecting each filter's _stream_mode declaration

## 4.5 — StreamCache: inference optimization

Under the hood, the Rust runtime uses a **StreamCache** to optimize inference:

1. **Filter states** are cached by `hash(config + training_data)` — loaded once
2. **Chunk results** are cached by `hash(config + state + chunk_data)` — if the same
   chunk passes through the same trained filter, the result is instant

This means: for FixedState filters during inference, repeated chunks cost nothing.

```
                    StreamCache
                    ┌─────────────────────┐
                    │ states:             │
                    │   scaler → {mean,std}│
                    │   model  → {weights} │
                    │                     │
                    │ chunks: (LRU)       │
                    │   hash_1 → result_1 │
                    │   hash_2 → result_2 │
                    └─────────────────────┘
```

---

**Next:** [05 — Advanced Patterns](./05_advanced_patterns.ipynb) — complex pipelines, multi-objective optimization, and pipeline+study integration.